# EveryQuery vs EIC: Multi-Duration Results

Single analysis notebook for the updated preprint. Loads the multi-duration EQ run
on MIMIC together with the matched EIC baseline, runs verification and statistical
comparisons, and generates all figures inline.

## Setup and Data Loading

In [ ]:
# Dataset to analyze. One of: "mimic", "columbia", "nwicu".
dataset = "mimic"

In [ ]:
from pathlib import Path

REPO = Path(".").resolve().parent
DATA = REPO / "data"

DATASET_CFG = {
    "mimic": {
        "run_dir": DATA / "outputs" / "2026-04-30" / "20-09-44",
        "metrics": "metrics_2026-05-02_12-07-02.parquet",
        "predictions": "predictions_2026-05-02_12-07-02.parquet",
        "eic": DATA / "baselines" / "eic" / "mimic" / "temporal-auc-results.parquet",
        "split_dir": DATA
        / "code-splits"
        / "mimic"
        / "split__seed42__poolbbd2128d07ed__ood7450ad537df2__id1ad4b1f900eb",
    },
    "columbia": {
        "run_dir": DATA / "outputs" / "2026-04-30" / "20-06-40",
        "metrics": "metrics_2026-05-02_03-31-43.parquet",
        "predictions": "predictions_2026-05-02_03-31-43.parquet",
        "eic": DATA / "baselines" / "eic" / "columbia" / "temporal-auc-results.parquet",
        "split_dir": DATA
        / "code-splits"
        / "columbia"
        / "split__seed42__poolc75b328e959b__ooda7759bfdd359__id71dda1371215",
    },
    "nwicu": {
        "run_dir": DATA / "outputs" / "2026-05-04" / "02-44-12",
        "metrics": "metrics_2026-05-04_15-33-29.parquet",
        "predictions": "predictions_2026-05-04_15-33-29.parquet",
        "eic": DATA / "baselines" / "eic" / "nwicu" / "temporal-auc-results.parquet",
        "split_dir": DATA
        / "code-splits"
        / "nwicu"
        / "split__seed42__pool3cc17c86f45e__oodb0e525c8bf23__id14c9331281ce",
    },
}

cfg = DATASET_CFG[dataset]
PREDS_DIR = cfg["run_dir"] / "preds"
METRICS_PARQUET = PREDS_DIR / cfg["metrics"]
PREDICTIONS_PARQUET = PREDS_DIR / cfg["predictions"]
EIC_PARQUET = cfg["eic"]
SPLIT_DIR = cfg["split_dir"]
FIG_DIR = Path("/Users/payalchandak/Desktop/EveryQuery/SD4H/figures") / dataset
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import hashlib
import re

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import yaml
from scipy import stats as sp_stats

matplotlib.rcParams.update(
    {
        "font.size": 11,
        "font.family": "sans-serif",
        "axes.linewidth": 1.0,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    }
)

%matplotlib inline


def _eic_slug(query: str, n_hash: int = 10, prefix_len: int = 24) -> str:
    """Reproduce EIC's task-name sanitization (sha256 + collapse non-word chars).

    The EIC baseline parquet uses these slugs as column suffixes (``AUC/<slug>``);
    EQ stores the raw query string. We compute slugs once here so the join can use
    a stable column key without round-tripping through file naming conventions.
    """
    h = hashlib.sha256(query.encode("utf-8")).hexdigest()[:n_hash]
    s = re.sub(r"[^A-Za-z0-9_]", "_", query.replace("//", "_"))
    return f"{s[:prefix_len]}__{h}"


with (SPLIT_DIR / "id_eval_codes.yaml").open() as f:
    id_codes = set(yaml.safe_load(f)["codes"])
with (SPLIT_DIR / "ood_eval_codes.yaml").open() as f:
    ood_codes = set(yaml.safe_load(f)["codes"])

eq = pl.read_parquet(METRICS_PARQUET).with_columns(
    pl.col("duration_days").cast(pl.Int64),
    pl.col("query")
    .map_elements(
        lambda q: "id" if q in id_codes else ("ood" if q in ood_codes else None),
        return_dtype=pl.Utf8,
    )
    .alias("bucket"),
    pl.col("query").map_elements(_eic_slug, return_dtype=pl.Utf8).alias("eic_slug"),
)

# Evaluation durations are derived from the run's metrics parquet so the rest of
# the notebook adapts automatically to datasets that ship a different set
# (e.g. NWICU has 5 durations vs 8 for MIMIC/Columbia). The task sampler draws
# durations continuously across the full grid at training time, so every eval
# duration is in-distribution.
DURATIONS = sorted(eq["duration_days"].unique().to_list())

eic_wide = pl.read_parquet(EIC_PARQUET).with_columns(
    (pl.col("duration").dt.total_microseconds() / (1e6 * 86400)).cast(pl.Int64).alias("duration_days")
)
eic = eic_wide.unpivot(
    on=[c for c in eic_wide.columns if c.startswith("AUC/")],
    index=["duration_days"],
    variable_name="eic_slug",
    value_name="eic_auroc",
).with_columns(pl.col("eic_slug").str.strip_prefix("AUC/"))

df = (
    eq.join(eic, on=["eic_slug", "duration_days"], how="inner")
    .filter(pl.col("occurs_auroc").is_not_null() & pl.col("eic_auroc").is_not_null())
    .with_columns((pl.col("occurs_auroc") - pl.col("eic_auroc")).alias("delta_auroc"))
    .drop("eic_slug")
)

print(f"EQ rows: {len(eq):,}, EIC rows: {len(eic):,}")
print(
    f"Joined: {len(df):,} (query, duration) pairs across "
    f"{df['query'].n_unique()} queries x {len(DURATIONS)} durations"
)

In [ ]:
# DROP OOD CODES
df = df.filter(pl.col("bucket") == "id")

## Data Verification

In [ ]:
nulls = {c: df[c].null_count() for c in df.columns if df[c].null_count() > 0}
print(f"Columns with nulls: {nulls}" if nulls else "No nulls")

id_n = df.filter(pl.col("bucket") == "id").shape[0]
ood_n = df.filter(pl.col("bucket") == "ood").shape[0]
print(f"Buckets: {id_n} ID, {ood_n} OOD")

print("\nOverall:")
for col in ("occurs_auroc", "eic_auroc", "delta_auroc"):
    arr = df[col].to_numpy()
    print(f"  {col}: mean={arr.mean():.4f}, median={np.median(arr):.4f}, std={arr.std():.4f}")

## Main Text Results

### EQ vs EIC Aggregate

In [ ]:
def bootstrap_ci(deltas: np.ndarray, n_boot: int = 10000, alpha: float = 0.05, seed: int = 42):
    rng = np.random.default_rng(seed)
    means = np.array([deltas[rng.integers(0, len(deltas), size=len(deltas))].mean() for _ in range(n_boot)])
    return np.percentile(means, 100 * alpha / 2), np.percentile(means, 100 * (1 - alpha / 2))


def compute_metrics(sub: pl.DataFrame) -> dict:
    deltas = sub["delta_auroc"].to_numpy()
    eq_arr = sub["occurs_auroc"].to_numpy()
    eic_arr = sub["eic_auroc"].to_numpy()
    n = len(deltas)
    wins = int((deltas > 0).sum())
    lo, hi = bootstrap_ci(deltas)
    if n >= 10:
        stat, p = sp_stats.wilcoxon(eq_arr, eic_arr, alternative="two-sided")
    else:
        stat, p = float("nan"), float("nan")
    return {
        "N_pairs": n,
        "EQ_wins": wins,
        "Win%": round(100 * wins / n, 1),
        "Mean_dAUC": round(float(deltas.mean()), 4),
        "Median_dAUC": round(float(np.median(deltas)), 4),
        "Mean_EQ_AUC": round(float(eq_arr.mean()), 4),
        "Mean_EIC_AUC": round(float(eic_arr.mean()), 4),
        "95%_CI": f"[{lo:.4f}, {hi:.4f}]",
        "Wilcoxon_stat": float(stat),
        "Wilcoxon_p": float(p),
    }


m = compute_metrics(df)
print("Overall EQ vs EIC comparison")
print(f"  N paired tasks x durations : {m['N_pairs']}")
print(f"  EQ wins                    : {m['EQ_wins']} / {m['N_pairs']}  ({m['Win%']}%)")
print(f"  Mean EQ AUC                : {m['Mean_EQ_AUC']}")
print(f"  Mean EIC AUC               : {m['Mean_EIC_AUC']}")
print(f"  Mean delta AUC (EQ - EIC)  : {m['Mean_dAUC']}")
print(f"  Median delta AUC           : {m['Median_dAUC']}")
print(f"  95% bootstrap CI (mean)    : {m['95%_CI']}")
print(f"  Wilcoxon signed-rank stat  : {m['Wilcoxon_stat']:.4f}")
print(f"  Wilcoxon signed-rank p     : {m['Wilcoxon_p']:.4e}")

rows_b = []
for dur in DURATIONS:
    sub = df.filter(pl.col("duration_days") == dur)
    if len(sub) == 0:
        continue
    met = compute_metrics(sub)
    if len(sub) >= 2:
        prev = sub["prevalence"].to_numpy()
        rho_eq, p_eq = sp_stats.spearmanr(sub["occurs_auroc"].to_numpy(), prev)
        rho_eic, p_eic = sp_stats.spearmanr(sub["eic_auroc"].to_numpy(), prev)
    else:
        rho_eq, p_eq, rho_eic, p_eic = (float("nan"),) * 4
    rows_b.append(
        {
            "Duration": dur,
            **met,
            "Rho_EQ_Prev": round(float(rho_eq), 4),
            "p_EQ_Prev": float(p_eq),
            "Rho_EIC_Prev": round(float(rho_eic), 4),
            "p_EIC_Prev": float(p_eic),
        }
    )

pl.DataFrame(rows_b)

### Loss Summary and Code-Bucket Breakdown

In [ ]:
n_total = len(df)
n_losses = df.filter(pl.col("delta_auroc") < 0).shape[0]
print(f"Total (query, duration) evaluations: {n_total:,}")
print(f"Losses (EIC > EQ): {n_losses:,} ({n_losses / n_total:.1%})")

queries = sorted(df["query"].unique().to_list())
win_all_durs = 0
lose_all_durs = []
for q in queries:
    sub = df.filter(pl.col("query") == q)
    deltas = sub["delta_auroc"]
    if (deltas > 0).all():
        win_all_durs += 1
    if (deltas < 0).all():
        lose_all_durs.append(q)

print(f"\nQueries winning at every evaluated duration:  {win_all_durs}/{len(queries)}")
print(f"Queries losing  at every evaluated duration: {len(lose_all_durs)}/{len(queries)}")
for q in lose_all_durs:
    print(f"  - {q}")

id_df = df.filter(pl.col("bucket") == "id")
ood_df = df.filter(pl.col("bucket") == "ood")
wins_id = int((id_df["delta_auroc"] > 0).sum())
wins_ood = int((ood_df["delta_auroc"] > 0).sum())
table = np.array([[wins_id, len(id_df) - wins_id], [wins_ood, len(ood_df) - wins_ood]])
or_code, p_code = sp_stats.fisher_exact(table, alternative="two-sided")

print("\nCode-OOD generalization (aggregate across all durations):")
print(f"  ID  queries: {wins_id}/{len(id_df)} wins ({wins_id / len(id_df):.1%})")
print(f"  OOD queries: {wins_ood}/{len(ood_df)} wins ({wins_ood / len(ood_df):.1%})")
print(f"  Fisher's exact: OR={or_code:.3f}, p={p_code:.4f}")

rows_c = []
for dur in DURATIONS:
    for bucket in ("id", "ood"):
        sub = df.filter((pl.col("duration_days") == dur) & (pl.col("bucket") == bucket))
        if len(sub) == 0:
            continue
        rows_c.append({"Duration": dur, "Code_Bucket": bucket, **compute_metrics(sub)})

pl.DataFrame(rows_c)

### Aggregate Code-OOD Comparison

In [ ]:
id_deltas = df.filter(pl.col("bucket") == "id")["delta_auroc"].to_numpy()
ood_deltas = df.filter(pl.col("bucket") == "ood")["delta_auroc"].to_numpy()

id_lo, id_hi = bootstrap_ci(id_deltas)
ood_lo, ood_hi = bootstrap_ci(ood_deltas)
stat_mw, p_mw = sp_stats.mannwhitneyu(id_deltas, ood_deltas, alternative="two-sided")

print("Aggregate Code-OOD Comparison (all durations pooled)")
print(
    f"  ID  queries: N={len(id_deltas):>4}, mean dAUC={id_deltas.mean():+.4f}  "
    f"95% CI [{id_lo:+.4f}, {id_hi:+.4f}]"
)
print(
    f"  OOD queries: N={len(ood_deltas):>4}, mean dAUC={ood_deltas.mean():+.4f}  "
    f"95% CI [{ood_lo:+.4f}, {ood_hi:+.4f}]"
)
print(f"  Difference (ID - OOD): {id_deltas.mean() - ood_deltas.mean():+.4f}")
print(f"  Mann-Whitney U={stat_mw:.1f}, p={p_mw:.4f}")

subsets = [
    ("Overall", df),
    ("ID queries", df.filter(pl.col("bucket") == "id")),
    ("OOD queries", df.filter(pl.col("bucket") == "ood")),
]
summary_rows = []
for label, sub in subsets:
    n = len(sub)
    summary_rows.append(
        {
            "Subset": label,
            "N_tasks": n,
            "N_queries": sub["query"].n_unique(),
            "Win_rate": round(float((sub["delta_auroc"] > 0).sum()) / n * 100, 1),
            "Mean_EQ_AUC": round(float(sub["occurs_auroc"].mean()), 4),
            "Mean_EIC_AUC": round(float(sub["eic_auroc"].mean()), 4),
            "Mean_dAUC": round(float(sub["delta_auroc"].mean()), 4),
        }
    )
pl.DataFrame(summary_rows)

In [ ]:
# Use duration as a continuous color (range 1-1826), keep color scheme,
# Decrease alpha of scatter points since they are overlapping a lot.
# show colorbar is now controlled by a bool flag
# ensure colors are visible (not appearing white or transparent)

from matplotlib import cm, colors

SHOW_COLORBAR = False  # set this flag to False if you do not want to show the colorbar

dur_array = df["duration_days"].to_numpy()
norm = colors.Normalize(vmin=30, vmax=1826)
cmap = cm.RdYlBu_r
rgb_colors = cmap(norm(dur_array))[:, :3]

rho_delta, p_delta = sp_stats.spearmanr(df["prevalence"].to_numpy(), df["delta_auroc"].to_numpy())

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sc = ax.scatter(
    df["prevalence"].to_numpy(),
    df["delta_auroc"].to_numpy(),
    c=rgb_colors,
    marker="o",
    s=36,
    alpha=0.65,
    edgecolors="none",
    zorder=3,
)

# Add colorbar for duration (optional)
if SHOW_COLORBAR:
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.03)
    cbar.set_label("Duration (days)", fontsize=12)

ax.axhline(0, color="0.5", linewidth=0.6, linestyle="-", zorder=1)
ax.set_xscale("log")
ax.set_xlabel("Prevalence (log scale)", fontsize=13)
ax.set_ylabel("EveryQuery win margin (AUC)", fontsize=13)

ax.text(
    0.97,
    0.95,
    f"$\\rho$ = {rho_delta:+.2f}, p = {p_delta:.1e}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=16,
    bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "0.7", "alpha": 0.9},
)

# No legend

fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "fig_prevalence_scatter.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "fig_prevalence_scatter.png", bbox_inches="tight", dpi=300)
plt.show()

prev_all = df["prevalence"].to_numpy()
delta_all = df["delta_auroc"].to_numpy()
eq_all = df["occurs_auroc"].to_numpy()
eic_all = df["eic_auroc"].to_numpy()

rho_d, p_d = sp_stats.spearmanr(prev_all, delta_all)
rho_e, p_e = sp_stats.spearmanr(prev_all, eq_all)
rho_b, p_b = sp_stats.spearmanr(prev_all, eic_all)
print("Pooled Spearman (prevalence vs metric, all durations):")
print(f"  rho(prev, dAUROC):    {rho_d:+.3f}  (p={p_d:.4f})")
print(f"  rho(prev, EQ AUROC):  {rho_e:+.3f}  (p={p_e:.4f})")
print(f"  rho(prev, EIC AUROC): {rho_b:+.3f}  (p={p_b:.4f})")

## Embedding Analysis

UMAP projection and pairwise cosine similarity of EQ query embeddings, sampled
across all 200 evaluation queries x 8 durations for a fixed pool of subjects.

In [ ]:
if dataset != "mmimic":
    print(f"Skipping embedding analysis for dataset={dataset}")
else:
    import hashlib

    import pyarrow.fs as pafs
    import pyarrow.parquet as pq
    import seaborn as sns
    import umap
    from sklearn.metrics.pairwise import cosine_similarity

    # Subjects x queries x durations = 200 x 8 = 1600 vectors per subject. The full
    # embeddings parquet is 40 GiB and impractical to keep on disk; instead we read
    # row-group-pruned slices directly from GCS and cache the filtered subset.
    #
    # subject_id is sorted across the parquet's 14k row groups, so a Pyarrow filter
    # on ``subject_id IN sampled_ids`` reads ~N_SUBJECTS row groups (~3 MB each)
    # rather than the full 40 GiB.
    N_SUBJECTS = 50
    RNG_SEED = 42

    EMBEDDINGS_GS_URI = (
        "every-query-runs/eq-mimic-5-years/2026-04-30/20-09-44/preds/"
        "predictions_2026-05-02_12-07-02.embeddings.parquet"
    )

    def _query_category(q: str) -> str:
        return q.split("//", 1)[0]

    def _load_embedding_subset(n_subjects: int, seed: int) -> pl.DataFrame:
        """Sample N subjects with full 8-duration coverage and pull their embeddings.

        Caches the filtered slice locally so reruns don't re-pay the GCS read; the
        cache key is a hash of (n_subjects, seed) so changing either invalidates.
        """
        cache_key = hashlib.sha256(f"{n_subjects}:{seed}".encode()).hexdigest()[:12]
        cache_path = PREDS_DIR / f"embed_subset__n{n_subjects}__seed{seed}__{cache_key}.parquet"
        if cache_path.exists():
            print(f"Loading cached embedding subset: {cache_path.name}")
            return pl.read_parquet(cache_path)

        full_keys = (
            pl.scan_parquet(PREDICTIONS_PARQUET)
            .group_by("subject_id")
            .agg(pl.col("duration_days").n_unique().alias("n_dur"))
            .filter(pl.col("n_dur") == len(DURATIONS))
            .select("subject_id")
            .collect()
        )
        rng = np.random.default_rng(seed)
        sampled = rng.choice(
            full_keys["subject_id"].to_numpy(),
            size=min(n_subjects, len(full_keys)),
            replace=False,
        )
        print(f"Sampling {len(sampled)} of {len(full_keys):,} subjects with full 8-duration coverage")

        print(f"Reading row-group-pruned slice from gs://{EMBEDDINGS_GS_URI}")
        fs = pafs.GcsFileSystem(anonymous=False)
        tbl = pq.read_table(
            EMBEDDINGS_GS_URI,
            filesystem=fs,
            columns=["subject_id", "query", "duration_days", "embedding"],
            filters=[("subject_id", "in", sampled.tolist())],
        )
        out = pl.from_arrow(tbl).with_columns(pl.col("duration_days").cast(pl.Int64))
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        out.write_parquet(cache_path)
        print(f"Wrote {len(out):,} rows to cache: {cache_path.name}")
        return out

    embed_df = _load_embedding_subset(N_SUBJECTS, RNG_SEED).with_columns(
        pl.col("query").map_elements(_query_category, return_dtype=pl.Utf8).alias("category"),
    )
    embed_mat = np.stack(embed_df["embedding"].to_numpy())

    print(f"Embedding matrix shape: {embed_mat.shape}")
    print(
        f"Subjects: {embed_df['subject_id'].n_unique()}, "
        f"queries: {embed_df['query'].n_unique()}, "
        f"durations: {sorted(embed_df['duration_days'].unique().to_list())}, "
        f"categories: {sorted(embed_df['category'].unique().to_list())}"
    )

In [ ]:
if dataset != "mmimic":
    print(f"Skipping embedding analysis for dataset={dataset}")
else:
    reducer = umap.UMAP(
        n_neighbors=30,
        min_dist=0.3,
        metric="cosine",
        random_state=RNG_SEED,
        n_jobs=1,
    )
    umap_2d = reducer.fit_transform(embed_mat)
    embed_df = embed_df.with_columns(
        pl.Series("umap_x", umap_2d[:, 0]),
        pl.Series("umap_y", umap_2d[:, 1]),
    )
    print(f"UMAP projection shape: {umap_2d.shape}")

    categories = sorted(embed_df["category"].unique().to_list())
    cat_palette = {c: plt.cm.tab10(i % 10) for i, c in enumerate(categories)}

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    kw = {"s": 6, "alpha": 0.6, "edgecolors": "none", "rasterized": True}

    ax = axes[0]
    cats = embed_df["category"].to_numpy()
    for c in categories:
        mask = cats == c
        ax.scatter(umap_2d[mask, 0], umap_2d[mask, 1], color=cat_palette[c], label=c, **kw)
    ax.legend(
        fontsize=8,
        markerscale=4,
        loc="best",
        frameon=False,
        handletextpad=0.3,
        borderpad=0.3,
        labelspacing=0.4,
    )
    ax.set_title("(a) By query category", fontsize=12)

    ax = axes[1]
    durs = embed_df["duration_days"].to_numpy()
    for d in DURATIONS:
        mask = durs == d
        ax.scatter(umap_2d[mask, 0], umap_2d[mask, 1], color=dur_palette[d], label=f"{d}d", **kw)
    ax.legend(fontsize=8, markerscale=4, loc="best", frameon=False, ncol=2, handletextpad=0.3, borderpad=0.3)
    ax.set_title("(b) By duration", fontsize=12)

    ax = axes[2]
    subjs = embed_df["subject_id"].to_numpy()
    prng = np.random.RandomState(0)
    u_subjs = np.unique(subjs)
    cmap = {s: plt.cm.hsv(prng.random()) for s in u_subjs}
    ax.scatter(umap_2d[:, 0], umap_2d[:, 1], c=[cmap[s] for s in subjs], **kw)
    ax.set_title("(c) By patient", fontsize=12)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.tight_layout()
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / "fig_embed_umap.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "fig_embed_umap.png", bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
if dataset != "mmimic":
    print(f"Skipping embedding analysis for dataset={dataset}")
else:
    MAX_PAIRS = 50_000
    rng = np.random.default_rng(RNG_SEED)

    queries_arr = embed_df["query"].to_numpy()
    dur_arr = embed_df["duration_days"].to_numpy()
    subj_arr = embed_df["subject_id"].to_numpy()

    # Build indices for the four similarity groups
    group_idx: dict[tuple[str, int], list[int]] = {}
    subj_idx: dict[int, list[int]] = {}
    for i in range(len(embed_df)):
        group_idx.setdefault((queries_arr[i], int(dur_arr[i])), []).append(i)
        subj_idx.setdefault(int(subj_arr[i]), []).append(i)

    def _cap(arr: np.ndarray, n: int = MAX_PAIRS) -> np.ndarray:
        if len(arr) <= n:
            return arr
        return rng.choice(arr, size=n, replace=False)

    # Same query, same duration, different patients
    sims = np.concatenate(
        [
            cosine_similarity(embed_mat[idxs])[np.triu_indices(len(idxs), k=1)]
            for idxs in group_idx.values()
            if len(idxs) > 1
        ]
    )
    sim_groups = {"Same query\nsame duration": _cap(sims)}

    # Same query, different duration, different patients
    sims_list = []
    queries_unique = sorted({q for q, _ in group_idx})
    for q in queries_unique:
        durs = sorted(d for (q2, d) in group_idx if q2 == q)
        for di, d1 in enumerate(durs):
            for d2 in durs[di + 1 :]:
                i1, i2 = group_idx[(q, d1)], group_idx[(q, d2)]
                cs = cosine_similarity(embed_mat[i1], embed_mat[i2])
                mask = subj_arr[i1][:, None] != subj_arr[i2][None, :]
                sims_list.append(cs[mask])
    sim_groups["Same query\ndiff duration"] = _cap(np.concatenate(sims_list))

    # Same patient, different queries
    sims_list = []
    for idxs in subj_idx.values():
        qs = np.unique(queries_arr[idxs])
        if len(qs) < 2:
            continue
        for ci, q1 in enumerate(qs):
            for q2 in qs[ci + 1 :]:
                r1 = [j for j in idxs if queries_arr[j] == q1]
                r2 = [j for j in idxs if queries_arr[j] == q2]
                sims_list.append(cosine_similarity(embed_mat[r1], embed_mat[r2]).ravel())
    sim_groups["Same patient\ndiff query"] = _cap(np.concatenate(sims_list))

    # Baseline: all three differ -- rejection sample then batched dot product
    all_keys = list(group_idx.keys())
    idx1: list[int] = []
    idx2: list[int] = []
    while len(idx1) < MAX_PAIRS:
        ka = rng.integers(0, len(all_keys), size=MAX_PAIRS * 2)
        kb = rng.integers(0, len(all_keys), size=MAX_PAIRS * 2)
        for a, b in zip(ka, kb, strict=False):
            k1, k2 = all_keys[a], all_keys[b]
            if k1[0] == k2[0] or k1[1] == k2[1]:
                continue
            i1 = int(rng.choice(group_idx[k1]))
            i2 = int(rng.choice(group_idx[k2]))
            if subj_arr[i1] == subj_arr[i2]:
                continue
            idx1.append(i1)
            idx2.append(i2)
            if len(idx1) >= MAX_PAIRS:
                break
    m1, m2 = embed_mat[idx1], embed_mat[idx2]
    sim_groups["Baseline\n(all differ)"] = np.sum(
        (m1 / np.linalg.norm(m1, axis=1, keepdims=True)) * (m2 / np.linalg.norm(m2, axis=1, keepdims=True)),
        axis=1,
    )

    group_order = list(sim_groups.keys())
    palette = ["#3A76AF", "#55A868", "#C44E52", "#8172B2"]
    plot_data = [{"Group": g, "Cosine similarity": float(v)} for g in group_order for v in sim_groups[g]]
    plot_df = pl.DataFrame(plot_data).to_pandas()

    fig, ax = plt.subplots(figsize=(6, 4.5))
    sns.violinplot(
        data=plot_df,
        x="Group",
        y="Cosine similarity",
        order=group_order,
        palette=palette,
        inner="box",
        linewidth=0.8,
        ax=ax,
        cut=0,
        saturation=0.85,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Cosine similarity")
    ax.tick_params(axis="x", labelsize=9)

    for i, g in enumerate(group_order):
        m = sim_groups[g].mean()
        ax.text(i, ax.get_ylim()[1] - 0.02, f"{m:.3f}", ha="center", va="top", fontsize=9, color="0.3")

    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_embed_cosine_sim.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "fig_embed_cosine_sim.png", bbox_inches="tight", dpi=300)
    plt.show()

    print(f"{'Group':<28} {'Mean':>7} {'Median':>7} {'Std':>7} {'N_pairs':>10} {'IQR':>7}")
    print("-" * 75)
    for g in group_order:
        v = sim_groups[g]
        iqr = float(np.percentile(v, 75) - np.percentile(v, 25))
        print(
            f"{g.replace(chr(10), ', '):<28} {v.mean():>7.4f} {np.median(v):>7.4f} {v.std():>7.4f} "
            f"{len(v):>10d} {iqr:>7.4f}"
        )

    print("\nMann-Whitney U (adjacent groups, one-sided greater):")
    for i in range(len(group_order) - 1):
        g1, g2 = group_order[i], group_order[i + 1]
        stat, p = sp_stats.mannwhitneyu(sim_groups[g1], sim_groups[g2], alternative="greater")
        print(f"  {g1.replace(chr(10), ', ')} > {g2.replace(chr(10), ', ')}: U={stat:.0f}, p={p:.2e}")

## Duration Monotonicity

For a fixed (patient, query), P(event by longer window) should be at least
P(event by shorter window). We measure the per-(patient, query) monotonicity
satisfaction rate (predictions are non-decreasing across consecutive durations)
on the held-out predictions parquet.

In [ ]:
preds_lf = pl.scan_parquet(PREDICTIONS_PARQUET).with_columns(pl.col("duration_days").cast(pl.Int64))

# Restrict to (subject, query) tuples evaluated at every duration -- partial
# coverage would silently bias the per-pair satisfaction counts.
full_keys = (
    preds_lf.group_by("subject_id", "query")
    .agg(pl.col("duration_days").n_unique().alias("n_dur"))
    .filter(pl.col("n_dur") == len(DURATIONS))
    .select("subject_id", "query")
    .collect()
)
preds_full = (
    preds_lf.join(full_keys.lazy(), on=["subject_id", "query"], how="semi")
    .select("subject_id", "query", "duration_days", "occurs_prob")
    .collect()
)
print(f"(subject, query) tuples with all {len(DURATIONS)} durations: {len(full_keys):,}")
print(f"Predictions after filtering: {len(preds_full):,}")

wide = preds_full.pivot(on="duration_days", index=["subject_id", "query"], values="occurs_prob").sort(
    "subject_id", "query"
)

consecutive_pairs = list(zip(DURATIONS[:-1], DURATIONS[1:], strict=True))

satisfaction_rows = []
for q in sorted(wide["query"].unique().to_list()):
    sub = wide.filter(pl.col("query") == q)
    n_subj = len(sub)
    any_violation = None
    pair_satisfied = {}
    for d_short, d_long in consecutive_pairs:
        col_short, col_long = str(d_short), str(d_long)
        violation_mask = sub[col_short] > sub[col_long]
        pair_satisfied[f"{d_short}->{d_long}"] = int(n_subj - violation_mask.sum())
        any_violation = violation_mask if any_violation is None else (any_violation | violation_mask)
    n_satisfied = int((~any_violation).sum())
    q_short = q if len(q) <= 45 else q[:42] + "..."
    satisfaction_rows.append(
        {
            "query": q_short,
            "n_subjects": n_subj,
            "n_satisfied": n_satisfied,
            "satisfaction_rate": round(n_satisfied / n_subj, 4),
            **pair_satisfied,
        }
    )

sat_df = pl.DataFrame(satisfaction_rows).sort("satisfaction_rate")
overall_rate = sat_df["n_satisfied"].sum() / sat_df["n_subjects"].sum()
print(
    f"\nOverall satisfaction rate (all consecutive pairs): "
    f"{sat_df['n_satisfied'].sum():,} / {sat_df['n_subjects'].sum():,} ({overall_rate:.1%})"
)
print()
print(sat_df.head(20))

In [ ]:
pair_cols = [f"{d_short}->{d_long}" for d_short, d_long in consecutive_pairs]
pair_labels = [f"{d_short}d → {d_long}d" for d_short, d_long in consecutive_pairs]
overall_rate = sat_df["n_satisfied"].sum() / sat_df["n_subjects"].sum()

fig, (ax_a, ax_b) = plt.subplots(
    1,
    2,
    figsize=(7.5, 3.5),
    gridspec_kw={"width_ratios": [1.6, 1], "wspace": 0.35},
)

rates = sat_df["satisfaction_rate"].to_numpy()
n_codes = len(rates)
bins = np.linspace(np.floor(rates.min() * 20) / 20, 1.0, 21)
ax_a.hist(rates, bins=bins, color="#4393C3", edgecolor="white", linewidth=0.5)
ax_a.axvline(overall_rate, color="0.3", ls="--", lw=1, zorder=5, label=f"Overall: {overall_rate:.1%}")
ax_a.set_xlabel("Satisfaction rate")
ax_a.set_ylabel(f"Number of codes (n={n_codes})")
ax_a.set_title("(a) Distribution of per-code satisfaction", fontsize=11, loc="left")
ax_a.legend(fontsize=8, frameon=False, loc="upper left")

pair_agg = [sat_df[pc].sum() / sat_df["n_subjects"].sum() for pc in pair_cols]
pair_agg.append(overall_rate)
x_labels = [*pair_labels, "All pairs"]
pair_palette = ["#4393C3"] * len(consecutive_pairs) + ["0.45"]

y_lo = max(0.0, min(pair_agg) - 0.02)
y_hi = 1.0 + (1.0 - y_lo) * 0.12
bars = ax_b.bar(
    range(len(x_labels)), pair_agg, color=pair_palette, edgecolor="white", linewidth=0.5, width=0.6
)
ax_b.set_xticks(range(len(x_labels)))
ax_b.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
ax_b.set_ylabel("Satisfaction rate")
ax_b.set_ylim(y_lo, y_hi)
ax_b.set_title("(b) By duration pair", fontsize=11, loc="left")

label_offset = (y_hi - y_lo) * 0.015
for bar, val in zip(bars, pair_agg, strict=True):
    ax_b.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + label_offset,
        f"{val:.1%}",
        ha="center",
        va="bottom",
        fontsize=8,
        color="0.15",
    )

fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "fig_monotonicity.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "fig_monotonicity.png", bbox_inches="tight", dpi=300)
plt.show()